# 00 — Acquire QCEW Data and build **September 2018 OMB** CBSA panel

- Downloads original inputs **only if missing**. Existing files are never silently replaced.
- Builds **only** 2018-vintage crosswalks and CBSA panels. Does not read or use another OMB delineation.
- Includes 2010–2023 historical QCEW for **outcome measurement**, but the forecasting notebook will **train only through June 2019** (the latest full QCEW quarter publicly released by February 2020).

**Historical-vintage limitation:** current BLS ZIP archives may contain revisions after 2020. This pipeline enforces the *first eligible reference-period release date* but cannot establish exact real-time vintage values without an archived 2019 first-release copy.

In [ ]:
# %pip install xlrd==2.0.2
from pathlib import Path
from datetime import datetime, timezone
import os, re, time, json, hashlib, warnings, zipfile
import pandas as pd
import numpy as np
import requests
import sys
import xlrd

print("Python:", sys.executable)
print("xlrd version:", xlrd.__version__)

# Optional override if launching from another folder: set RL_PROJECT_ROOT
ROOT=Path(os.environ['RL_PROJECT_ROOT']).expanduser().resolve() if os.environ.get('RL_PROJECT_ROOT') else next(
    (p for p in [Path.cwd(),*Path.cwd().parents] if (p/'code').is_dir() and (p/'data').is_dir()),None)
if ROOT is None:
    ROOT=Path.cwd().parent if Path.cwd().name=='code' else Path.cwd()
ROOT=ROOT.resolve()
RAW=ROOT/'data'/'raw'; PROC=ROOT/'data'/'processed'; OUT=ROOT/'output'
for p in (RAW/'qcew',RAW/'geography',PROC,OUT):p.mkdir(parents=True,exist_ok=True)
GEO_VINTAGE='2018-09'
OMB_URL=('https://www2.census.gov/programs-surveys/metro-micro/geographies/'
         'reference-files/2018/delineation-files/list1_Sep_2018.xls')
QCEW_URL='https://data.bls.gov/cew/data/files/{year}/csv/{year}_qtrly_by_area.zip'
EXPECTED_DATES=pd.date_range('2010-01-01','2023-12-01',freq='MS')
COUNTY_FILE=PROC/'qcew_county_monthly_2010_2023.csv'
CROSSWALK_FILE=PROC/'county_cbsa_crosswalk_sep2018.csv'
PANEL_FILE=PROC/'qcew_cbsa_monthly_2010_2023_omb2018.csv'

def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):digest.update(block)
    return digest.hexdigest()

def download_if_missing(url,dest,kind='file'):
    dest=Path(dest)
    if dest.exists():
        if dest.stat().st_size==0: raise ValueError(f'Empty cached file: {dest}. Remove it to retry.')
        if kind=='zip' and not zipfile.is_zipfile(dest):
            raise ValueError(f'Cached ZIP is corrupt: {dest}. Remove it manually to retry.')
        print('REUSE',dest.relative_to(ROOT))
        return dest
    dest.parent.mkdir(parents=True,exist_ok=True)
    temp=dest.with_name(dest.name+'.part')
    for attempt in range(3):
        try:
            print('DOWNLOAD',url)
            with requests.get(url,stream=True,timeout=(30,180),headers={
                'User-Agent':'regional-labor-resilience research replication'}) as r:
                r.raise_for_status()
                with temp.open('wb') as f:
                    for block in r.iter_content(chunk_size=1024*1024):
                        if block:f.write(block)
            if temp.stat().st_size==0 or (kind=='zip' and not zipfile.is_zipfile(temp)):
                raise ValueError('Downloaded file is empty or invalid ZIP')
            temp.replace(dest)
            return dest
        except (requests.RequestException,OSError,ValueError) as exc:
            temp.unlink(missing_ok=True)
            if attempt==2:raise RuntimeError(f'Download failed: {url}') from exc
            time.sleep(2**attempt)
    return dest

print('Project root:',ROOT)

  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
Using cached xlrd-2.0.2-py2.py3-none-any.whl (96 kB)
Note: you may need to restart the kernel to use updated packages.
Python: c:\Users\adamt\AppData\Local\Programs\Python\Python312\python.exe
xlrd version: 2.0.2
Project root: C:\Users\adamt\OneDrive\Documents\Directed Practicum\regional-labor-resilience



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Read September 2018 county-to-CBSA delineations

Uses the official historical Census Bureau `.xls` file. Install `xlrd` for this legacy format. A pre-existing derived 2018 crosswalk CSV is reused if present; no other vintage is consulted.

In [5]:
if CROSSWALK_FILE.exists():
    cw=pd.read_csv(CROSSWALK_FILE,dtype={'cbsa_code':'string','county_fips':'string','state_fips':'string'})
    print('REUSE',CROSSWALK_FILE.relative_to(ROOT))
else:
    xls=download_if_missing(OMB_URL,RAW/'geography'/'list1_Sep_2018.xls')
    try:raw=pd.read_excel(xls,sheet_name=0,header=None,dtype=str,engine='xlrd')
    except ImportError as e:raise ImportError('Install xlrd: pip install xlrd') from e
    def clean_label(x):return re.sub(r'\s+',' ',str(x).strip().lower())
    header=next((i for i in range(min(60,len(raw)))
        if any('cbsa code' in clean_label(x) for x in raw.iloc[i])
        and any('state' in clean_label(x) and 'fips' in clean_label(x) for x in raw.iloc[i])
        and any('county' in clean_label(x) and 'fips' in clean_label(x) for x in raw.iloc[i])),None)
    if header is None:raise ValueError('Official OMB workbook header not recognized; inspect first 60 rows.')
    body=raw.iloc[header+1:].copy()
    body.columns=[str(x).strip() for x in raw.iloc[header]]
    def col(*tokens):
        matches=[s for s in body.columns if all(t in clean_label(s) for t in tokens)]
        if not matches:raise ValueError(f'Missing OMB header {tokens}; found: {list(body.columns)}')
        return matches[0]
    def norm(s,n):return s.astype('string').str.strip().str.replace(r'\.0$','',regex=True).str.zfill(n)
    cw=pd.DataFrame({
        'cbsa_code':norm(body[col('cbsa','code')],5),
        'state_fips':norm(body[col('fips','state')],2),
        'county_code':norm(body[col('fips','county')],3),
        'cbsa_type':body[col('metropolitan','micropolitan')].astype('string').str.strip()})
    cw['county_fips']=cw.state_fips+cw.county_code
    cw=cw.loc[cw.cbsa_code.str.fullmatch(r'\d{5}',na=False)
              & cw.county_fips.str.fullmatch(r'\d{5}',na=False)
              & cw.cbsa_type.str.contains('metropolitan|micropolitan',case=False,na=False)]
    cw=cw.drop_duplicates(['cbsa_code','county_fips'])
    cw.to_csv(CROSSWALK_FILE,index=False)
cw['cbsa_code']=cw.cbsa_code.astype('string').str.zfill(5)
cw['county_fips']=cw.county_fips.astype('string').str.zfill(5)
cw['state_fips']=cw.state_fips.astype('string').str.zfill(2)
if cw.empty or cw.duplicated('county_fips').any():raise ValueError('Empty/ambiguous 2018 CBSA crosswalk')
if cw.cbsa_code.nunique()<2:raise ValueError('Too few CBSAs; crosswalk appears incomplete')
# Confirm CT's historical eight counties are represented in the published 2018 delineation.
ct_expected={'09001','09003','09005','09007','09009','09011','09013','09015'}
ct=set(cw.loc[cw.state_fips.eq('09'),'county_fips'])
if not ct_expected.issubset(ct) and os.environ.get('RL_SYNTHETIC_TEST')!='1':
    raise ValueError('2018 crosswalk does not contain all eight historical Connecticut counties')
print('2018 CBSA count:',cw.cbsa_code.nunique(),'| CT historical county codes:',sorted(ct))

REUSE data\raw\geography\list1_Sep_2018.xls
2018 CBSA count: 938 | CT historical county codes: ['09001', '09003', '09005', '09007', '09009', '09011', '09013', '09015']


## 2. Build or reuse county-level QCEW monthly employment

If the geographically neutral county CSV exists, no ZIPs are downloaded. Otherwise, download each required year **only when the ZIP is missing**, and extract all-ownership (`own_code=0`), all-industry (`industry_code=10`) county monthly employment. Suppression and missing data remain missing.

In [6]:
if COUNTY_FILE.exists():
    print('REUSE existing county-level CSV:',COUNTY_FILE.relative_to(ROOT))
    county=pd.read_csv(COUNTY_FILE,dtype={'county_fips':'string'},parse_dates=['date'])
else:
    yearly=[]
    cols=['area_fips','year','qtr','own_code','industry_code','disclosure_code',
          'month1_emplvl','month2_emplvl','month3_emplvl']
    for year in range(2010,2024):
        archive=download_if_missing(QCEW_URL.format(year=year),
            RAW/'qcew'/f'{year}_qtrly_by_area.zip',kind='zip')
        yearparts=[]
        with zipfile.ZipFile(archive) as z:
            for name in z.namelist():
                if not name.lower().endswith('.csv'):continue
                with z.open(name) as stream:
                    for chunk in pd.read_csv(stream,dtype=str,usecols=lambda k:k in cols,
                                              chunksize=100000,low_memory=False,encoding='latin1'):
                        if not set(cols).issubset(chunk.columns):
                            raise ValueError(f'QCEW columns missing in {archive.name}/{name}')
                        for k in ['area_fips','year','qtr','own_code','industry_code','disclosure_code']:
                            chunk[k]=chunk[k].astype('string').str.strip()
                        m=(chunk.area_fips.str.fullmatch(r'\d{5}',na=False) &
                           ~chunk.area_fips.str.endswith('000',na=False) &
                           chunk.own_code.eq('0') & chunk.industry_code.eq('10') &
                           chunk.year.eq(str(year)))
                        sub=chunk.loc[m].copy()
                        if len(sub):yearparts.append(sub)
        if not yearparts:raise ValueError(f'No total-covered county data in {archive.name}')
        qq=pd.concat(yearparts,ignore_index=True)
        if qq.duplicated(['area_fips','qtr']).any():
            raise ValueError(f'Duplicate county/quarter total-industry rows for {year}')
        for i in (1,2,3):
            z=qq[['area_fips','year','qtr','disclosure_code',f'month{i}_emplvl']].copy()
            z['date']=pd.to_datetime(dict(
                year=pd.to_numeric(z.year),
                month=(pd.to_numeric(z.qtr)-1)*3+i,day=1))
            z['employment']=pd.to_numeric(z[f'month{i}_emplvl'],errors='coerce')
            z.loc[z.disclosure_code.fillna('').str.upper().eq('N'),'employment']=np.nan
            yearly.append(z.rename(columns={'area_fips':'county_fips'})[
                ['county_fips','date','employment']])
        print('Extracted year',year)
    county=pd.concat(yearly,ignore_index=True)
    county.to_csv(COUNTY_FILE,index=False)
county['county_fips']=county.county_fips.astype('string').str.zfill(5)
county['date']=pd.to_datetime(county.date).dt.to_period('M').dt.to_timestamp()
county['employment']=pd.to_numeric(county.employment,errors='coerce')
if county.duplicated(['county_fips','date']).any():raise ValueError('Duplicate county-month observations')
if county.date.min()!=EXPECTED_DATES.min() or county.date.max()!=EXPECTED_DATES.max():
    raise ValueError('County panel must span 2010-01 through 2023-12')
print('County monthly rows:',len(county),'| Unique counties:',county.county_fips.nunique())

REUSE existing county-level CSV: data\processed\qcew_county_monthly_2010_2023.csv
County monthly rows: 550704 | Unique counties: 3278


## 3. Aggregate counties to 2018 CBSAs, with strict complete-coverage checks

Every constituent county must have a valid employment observation for the month. The primary sample retains CBSAs with all 168 months complete. **Sample size is recalculated** rather than hard-coded. Excluded regions and Connecticut coverage are audited.

In [7]:
expected=cw.groupby('cbsa_code').county_fips.nunique().rename('expected_counties')
joined=cw[['cbsa_code','county_fips','cbsa_type']].merge(
    county[['county_fips','date','employment']],on='county_fips',how='left',validate='one_to_many')
g=joined.groupby(['cbsa_code','date']).employment
coverage=pd.DataFrame({'employment':g.sum(min_count=1),'observed_counties':g.count()}).reset_index()
idx=pd.MultiIndex.from_product([sorted(cw.cbsa_code.unique()),EXPECTED_DATES],names=['cbsa_code','date'])
coverage=coverage.set_index(['cbsa_code','date']).reindex(idx).reset_index()
coverage=coverage.merge(expected,on='cbsa_code',validate='many_to_one')
coverage['observed_counties']=coverage.observed_counties.fillna(0).astype(int)
coverage['complete']=coverage.observed_counties.eq(coverage.expected_counties)
coverage.loc[~coverage.complete,'employment']=np.nan
ca=coverage.groupby('cbsa_code').agg(complete_months=('complete','sum')).reset_index()
ca['missing_months']=len(EXPECTED_DATES)-ca.complete_months
ca.to_csv(OUT/'omb2018_cbsa_coverage_audit.csv',index=False)
keep=set(ca.loc[ca.missing_months.eq(0),'cbsa_code'])
ca.loc[~ca.cbsa_code.isin(keep)].to_csv(OUT/'omb2018_cbsa_excluded.csv',index=False)
if not keep:raise ValueError('No CBSAs with 168 valid months: check QCEW source and crosswalk')
types=cw[['cbsa_code','cbsa_type']].drop_duplicates('cbsa_code')
panel=(coverage.loc[coverage.cbsa_code.isin(keep),['cbsa_code','date','employment']]
       .merge(types,on='cbsa_code',validate='many_to_one')
       .sort_values(['cbsa_code','date']).reset_index(drop=True))
assert panel.groupby('cbsa_code').size().eq(168).all()
assert panel.employment.notna().all() and (panel.employment>0).all()
panel.to_csv(PANEL_FILE,index=False)
ct_cbsas=set(cw.loc[cw.state_fips.eq('09'),'cbsa_code'])
manifest={
    'geography_vintage':'2018-09',
    'information_cutoff':'2020-02-29',
    'latest_QCEW_full_data_available_as_of_cutoff':'2019-Q2 (released 2019-12-04)',
    'qcew_forecasting_training_end':'2019-06-01',
    'later_QCEW_records_use':'outcome_and_diagnostics_only',
    'historical_vintage_limitation':'Current BLS QCEW downloadable archives may include subsequent revisions',
    'crosswalk_csv':str(CROSSWALK_FILE.relative_to(ROOT)),
    'crosswalk_sha256':sha256(CROSSWALK_FILE),
    'county_csv':str(COUNTY_FILE.relative_to(ROOT)),
    'county_sha256':sha256(COUNTY_FILE),
    'panel_csv':str(PANEL_FILE.relative_to(ROOT)),
    'panel_sha256':sha256(PANEL_FILE),
    'n_cbsa':int(panel.cbsa_code.nunique()),
    'ct_cbsa_count_in_crosswalk':len(ct_cbsas),
    'ct_cbsa_count_in_panel':len(ct_cbsas.intersection(keep)),
    'generated_utc':datetime.now(timezone.utc).isoformat()
}
(PROC/'omb2018_geography_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('2018 panel:',PANEL_FILE.relative_to(ROOT),'| CBSAs:',manifest['n_cbsa'])
print('Connecticut CBSAs in panel:',manifest['ct_cbsa_count_in_panel'])

2018 panel: data\processed\qcew_cbsa_monthly_2010_2023_omb2018.csv | CBSAs: 936
Connecticut CBSAs in panel: 6
